In [ ]:
import PyPDF2
import re

# Function to extract numbers under specific conditions
def extract_conditions_data(text, condition_name):
    start_index = text.find(condition_name)
    if start_index == -1:
        return None
    end_index = text.find("Conditions", start_index + len(condition_name))
    if end_index == -1:
        end_index = len(text)
    condition_text = text[start_index:end_index]
    lines = condition_text.splitlines()
    
    # Regex to find numbers that are not followed by unwanted characters
    number_pattern = re.compile(r'\b\d+(\.\d+)?\b(?![^\s])')
    
    # Extract numbers under a, b, c, ... p
    data = []
    for line in lines:
        if any(char.isdigit() for char in line):
            matches = number_pattern.findall(line)
            clean_numbers = [match for match in matches if re.match(r'^\d+(\.\d+)?$', match)]
            if clean_numbers:
                data.append(clean_numbers)
    return data

# Load the PDF
pdf_path = 'HOF_CD/745430_SI.pdf'
pdf_file = open(pdf_path, 'rb')
pdf_reader = PyPDF2.PdfReader(pdf_file)

# Extract text from each page
text = ""
for page_num in range(len(pdf_reader.pages)):
    page = pdf_reader.pages[page_num]
    text += page.extract_text()


# text


# # Extract the required data
heating_data = extract_conditions_data(text, "Annual Heating, Humidification, and Ventilation Design Conditions")
cooling_data = extract_conditions_data(text, "Annual Cooling, Dehumidification, and Enthalpy Design Conditions")
extreme_annual_data = extract_conditions_data(text, "Extreme Annual Design Conditions")

# # Close the PDF file
# pdf_file.close()

heating_data

# # Print the extracted data
# print("Heating, Humidification, and Ventilation Design Conditions Data:")
# print(heating_data)
# print("\nCooling, Dehumidification, and Enthalpy Design Conditions Data:")
# print(cooling_data)
# print("\nExtreme Annual Design Conditions Data (DB Only):")
# print(extreme_annual_data)


[]

In [2]:
import PyPDF2
import re

# Modified function to extract the full section as a list of lines

# Load the PDF
wmo = 722167

pdf_path = f'HOF_CD/{wmo}_SI.pdf'
with open(pdf_path, 'rb') as pdf_file:
    pdf_reader = PyPDF2.PdfReader(pdf_file)
    text = ""
    for page in pdf_reader.pages:
        text += page.extract_text()

# Now you can use heating_data, cooling_data, extreme_annual_data
# For example:
text

'2021 ASHRAE Handbook — Fundamentals (SI)© 2021 ASHRAE, Inc.ORANGE COUNTY AP, VA, USAWMO:  722167\nLat: 38.247N Lon: 78.046W Elev: 143 StdP:  99.62 Time Zone: -5.00 (NAE)Period:  04-19 WBAN:  03718Annual Heating, Humidification, and Ventilation Design ConditionsColdest\nMonthHeating DBHumidification DP/MCDB and HR Coldest Month WS/MCDB MCWS/PCWD\nto 99.6% DB WSF 99.6% 99% 0.4% 1%\n99.6% 99% DP HR MCDB DP HR MCDB WS MCDB WS MCDB MCWS PCWD(a)(b)(c)(d)(e)(f)(g)(h)(i)(j)(k)(l)(m)(n)(o)(p)(1) 1 -10.2 -7.6 -17.9 0.8 -6.2 -15.3 1.0 -3.7 9.2 8.6 8.3 6.4 0.9 310 0.400 (1)Annual Cooling, Dehumidification, and Enthalpy Design ConditionsHottest\nMonthHottest\nMonth\nDB RangeCooling DB/MCWBEvaporation WB/MCDBMCWS/PCWD\nto 0.4% DB 0.4% 1% 2% 0.4% 1% 2%\nDB MCWB DB MCWB DB MCWB WB MCDB WB MCDB WB MCDB MCWS PCWD(a)(b)(c)(d)(e)(f)(g)(h)(i)(j)(k)(l)(m)(n)(o)(p)(2) 7 11.3 34.0 24.4 32.6 24.2 31.3 23.8 27.0 30.2 25.8 30.1 25.1 29.6 2.5 180 (2)\nDehumidification DP/MCDB and HREnthalpy/MCDBExtreme\nMax WB0.

In [3]:

import re

def extract_design_conditions(epw_source, raw_text):
    # Normalize the text
    raw_text = raw_text.replace('\n', ' ').replace('©', '').replace('—', '--')

    # Header info
    source_info = f"{epw_source} - Chapter 14 Climatic Design Information"
    result = ['DESIGN CONDITIONS', '1', source_info, '']

    # Utility function to extract values
    def extract_section(tag, limit=None):
        pattern = rf'\({tag}\)\s*(.*?)\s*\({tag}\)'
        match = re.search(pattern, raw_text)
        if not match:
            return []
        values = match.group(1).strip().split()
        # Remove the leading number (station month) if it's a single integer
        if values and re.fullmatch(r'\d+', values[0]):
            values = values[1:]
        # Remove labels like DB, WB
        values = [v for v in values if v not in ['DB', 'WB']]
        if limit:
            values = values[:limit]
        return values

    # Extract each section
    heating_values = extract_section('1')
    cooling_values = extract_section('2')
    dehum_values = extract_section('3')
    extreme_db_values = extract_section('4', limit=12)
    extreme_wb_values = extract_section('5', limit=12)

    # Extract the first number in cooling section as month ID
    cooling_month = ''
    m = re.search(r'\(2\)\s*(\d+)', raw_text)
    if m:
        cooling_month = m.group(1)

    # Assemble result
    result += ['Heating', '1'] + heating_values
    result += ['Cooling', cooling_month] + cooling_values + dehum_values
    result += ['Extremes'] + extreme_db_values + extreme_wb_values

    return ','.join(result)

# Example usage:
raw_string = text
epw_line = extract_design_conditions("2021 ASHRAE Handbook -- Fundamentals", raw_string)
epw_line

'DESIGN CONDITIONS,1,2021 ASHRAE Handbook -- Fundamentals - Chapter 14 Climatic Design Information,,Heating,1,-10.2,-7.6,-17.9,0.8,-6.2,-15.3,1.0,-3.7,9.2,8.6,8.3,6.4,0.9,310,0.400,Cooling,7,11.3,34.0,24.4,32.6,24.2,31.3,23.8,27.0,30.2,25.8,30.1,25.1,29.6,2.5,180,26.1,21.9,28.3,24.9,20.4,27.7,23.7,18.8,26.8,85.3,30.2,80.4,31.0,77.1,28.9,30.4,Extremes,7.8,6.7,5.5,-14.8,36.5,3.0,1.5,-17.0,37.6,-18.8,38.5,-20.4,-15.3,27.5,2.9,1.1,-17.4,28.3,-19.1,28.9,-20.8,29.5,-22.9,30.3'

In [ ]:
DESIGN CONDITIONS,1,2021 ASHRAE Handbook -- Fundamentals - Chapter 14 Climatic Design Information,,Heating,1,-13.8,-11.3,-19.4,0.7,-10.9,-17.1,0.9,-9.1,14.6,2.6,12.9,2.0,4.7,0,0.607,Cooling,7,12.8,37.5,21.0,36.1,21.1,34.1,21.0,23.7,31.3,23.0,30.5,22.4,30.0,6.2,190,21.8,17.7,26.4,21.1,16.9,25.9,20.2,16.0,25.3,73.8,31.2,71.2,30.4,68.8,30.2,26.2,Extremes,13.1,11.8,10.7,-18.5,39.6,3.4,1.8,-20.9,40.9,-22.9,41.9,-24.9,42.9,-27.4,44.2

In [19]:
# Provided list
data_list = ['Annual Heating, Humidification, and Ventilation Design ConditionsColdest', 
             'MonthHeating DBHumidification DP/MCDB and HR Coldest Month WS/MCDB MCWS/PCWD',
             'to 99.6% DB WSF 99.6% 99% 0.4% 1%', 
             '99.6% 99% DP HR MCDB DP HR MCDB WS MCDB WS MCDB MCWS PCWD(a)(b)(c)(d)(e)(f)(g)(h)(i)(j)(k)(l)(m)(n)(o)(p)(1) 1 -7.3 -4.9 -12.0 1.3 -5.1 -10.1 1.6 -3.4 20.1 1.4 18.3 2.1 1.8 230 0.895 (1)Annual Cooling, Dehumidification, and Enthalpy Design ']

# String to be processed
data_string = ' '.join(data_list)

# Extracting the values between (n)(o)(p)(1) and (1)Annual Cooling
start = data_string.find('(n)(o)(p)(1)') + len('(n)(o)(p)(1)')
end = data_string.find('(1)Annual Cooling')

# Extract the relevant part and clean it up
extracted_values = data_string[start:end].strip()

# Convert to a list of elements
elements = extracted_values.split()

# Join elements with commas
comma_separated_string = ', '.join(elements)

print(comma_separated_string)

1, -7.3, -4.9, -12.0, 1.3, -5.1, -10.1, 1.6, -3.4, 20.1, 1.4, 18.3, 2.1, 1.8, 230, 0.895


In [7]:
import PyPDF2
import re

def extract_design_conditions(pdf_path):
    # Read the PDF content
    with open(pdf_path, 'rb') as pdf_file:
        pdf_reader = PyPDF2.PdfReader(pdf_file)
        full_text = ""
        for page in pdf_reader.pages:
            full_text += page.extract_text()

    # Function to extract numbers from a regex match
    def format_values(match):
        if match:
            # Extract all floating point or integer numbers including negatives
            numbers = re.findall(r"-?\d+\.\d+", match.group(1))
            return ', '.join(numbers)
        return None

    # Search for the data rows
    heating_match = re.search(r"\( 1 \)\s*([\d\s\.\-]+)\( 1 \)", full_text)
    cooling_match = re.search(r"\( 2 \)\s*([\d\s\.\-]+)\( 2 \)", full_text)
    extreme_match = re.search(r"\( 4 \)\s*([\d\s\.\-]+)DB", full_text)  # DB line only

    # Extract and format values
    heating_data = format_values(heating_match)
    cooling_data = format_values(cooling_match)
    extreme_data = format_values(extreme_match)

    return heating_data, cooling_data, extreme_data

# Example usage
pdf_file_path = 'HOF_CD/745430_SI.pdf'  # Replace with your actual path
heating, cooling, extreme_db = extract_design_conditions(pdf_file_path)

print("Heating Design Data:\n", heating)
print("\nCooling Design Data:\n", cooling)
print("\nExtreme Annual Dry Bulb Data:\n", extreme_db)


Heating Design Data:
 None

Cooling Design Data:
 None

Extreme Annual Dry Bulb Data:
 None
